# 06 — Diagnóstico dos erros no teste (seção 23 do v5)

**Lê:** `previsoes_teste` (05), `selecao_modelo.json`, `perfis_treino`, `panel_base` até `TRAIN_TARGET_END`, teste (trava, finalidade *diagnostico*).  
**Grava:** tabelas e figuras `diagnostico_*`.

**Correção em relação ao v5:** o modelo do diagnóstico principal é o escolhido em `selecao_modelo.json` (parâmetro `DIAG_MODELO_PRINCIPAL` do config). No v5 ele estava fixo em `CatBoost_MAE`, embora a seleção tenha escolhido `LightGBM_MAE_Exog`.

In [ ]:
# === SETUP (idêntico em todos os notebooks; só muda NOTEBOOK) ===
NOTEBOOK = "06_diagnostico"
RAIZ = "/content/drive/MyDrive/tcc_fluxo"      # Colab. Localmente: caminho da pasta do projeto (ou deixe None)

import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q -r {RAIZ}/requirements.txt
except ImportError:                              # fora do Colab
    RAIZ = os.environ.get("TCC_RAIZ") or None
    if RAIZ is None:
        from pathlib import Path
        RAIZ = str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").is_file()))
sys.path.insert(0, f"{RAIZ}/src")
%load_ext autoreload
%autoreload 2

from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tcc_fluxo import iniciar
C, A = iniciar(RAIZ, notebook=NOTEBOOK)

In [ ]:
from tcc_fluxo import diagnostico as D, metricas as M, graficos

sel = A.carregar_json("selecao_modelo")
MAIN = sel["modelo_escolhido"] if C.DIAG_MODELO_PRINCIPAL == "escolhido" else C.DIAG_MODELO_PRINCIPAL
comparison = A.carregar_df("previsoes_teste")
test = A.carregar_teste(finalidade="diagnostico")
hist = A.carregar_historico_treino()
profile = A.carregar_df("perfis_treino").set_index("seller_id")
scale_final = M.seller_scales(hist, C.TRAIN_TARGET_END)

MODELS = [m for m in D.ALL_DIAG if f"pred_{m}" in comparison.columns]
FOCUS = [m for m in D.FOCUS if m in MODELS]
assert MAIN in MODELS, f"{MAIN} sem previsões no teste"
thr = D.limiares(hist)
print(f"Modelo do diagnóstico principal: {MAIN}")
print(f"Pico > R$ {thr['PEAK_THRESHOLD']:,.2f} | sinal ≥ R$ {thr['SIGNAL_THRESHOLD']:,.2f} (P90/P10 da receita positiva no treino)")
diag = D.base_diagnostico(comparison, profile, test, MODELS, thr)

### 23.1 Diagnóstico resumido de todos os modelos

In [ ]:
geral = D.diagnostico_geral(diag, MODELS, thr["SIGNAL_THRESHOLD"])
A.tabela(geral, "diagnostico_geral_modelos.csv")
display(geral.set_index("Modelo").style.format("{:,.2f}"))

### 23.2 Semanas sem × com receita e faixas de valor

In [ ]:
diag["grupo"] = np.where(diag["y"] > 0, "com receita", "sem receita")
zp = D.group_table(diag, "grupo", list(dict.fromkeys([MAIN, "Naive"])))
faixas = D.group_table(diag, "faixa_real", list(dict.fromkeys([m for m in ["LightGBM_MAE", "LightGBM_MAE_Exog", "CatBoost_MAE",
                                                         "CatBoost_MAE_Exog", MAIN, "Naive", "CatBoost_Tweedie"] if m in MODELS])))
A.tabela(zp, "diagnostico_principal_zero_positivo.csv"); A.tabela(faixas, "diagnostico_faixas_valor.csv")
display(zp); display(faixas)
z = diag["y"] == 0
print(f"Em semanas sem receita, {MAIN} prevê exatamente 0 em {diag.loc[z, f'pred_{MAIN}'].eq(0).mean():.1%} "
      f"(Naive: {diag.loc[z, 'pred_Naive'].eq(0).mean():.1%}).")

### 23.3 Erros por vendedor

In [ ]:
sel_models = list(dict.fromkeys(FOCUS + [MAIN]))
ds = D.por_vendedor(diag, MAIN, sel_models, scale_final)
A.tabela(ds, "diagnostico_principal_por_vendedor.csv")
sk = ds["skill_vs_naive"].dropna()
print(f"Vendedores com viés negativo no total do teste: {(ds[f'vies_{MAIN}'] < 0).mean():.1%}")
print(f"Vendedores em que {MAIN} erra menos que o Naive: {(sk > 0).sum()} de {len(sk)} ({(sk > 0).mean():.1%})")
print(f"Vendedores sem nenhuma venda no teste: {int((ds['real_total'] == 0).sum())}")
conc = ds.sort_values(f"erro_abs_total_{MAIN}", ascending=False)
tot = conc[f"erro_abs_total_{MAIN}"].sum()
print(f"Concentração: 10 vendedores = {conc[f'erro_abs_total_{MAIN}'].head(10).sum() / tot:.1%} do erro; "
      f"10% dos vendedores = {conc[f'erro_abs_total_{MAIN}'].head(int(np.ceil(0.1 * len(conc)))).sum() / tot:.1%}")
fig = graficos.erro_por_vendedor(ds, MAIN); A.figura(fig, "diagnostico_erro_por_vendedor.png"); plt.show()

### 23.4–23.5 Erros por perfil e por semana

In [ ]:
dp, zero_ref = D.por_perfil(diag, MODELS, thr["SIGNAL_THRESHOLD"])
A.tabela(dp, "diagnostico_por_perfil.csv")
print("Referência trivial — MAE de prever zero sempre:"); display(zero_ref.to_frame().T)
display(dp.loc[dp["modelo"].isin(FOCUS)])

dw = D.por_semana(diag, MODELS)
A.tabela(dw, "diagnostico_por_semana.csv")
print(f"Semanas em que {MAIN} erra menos que o Naive: {(dw[f'MAE_{MAIN}'] < dw['MAE_Naive']).sum()} de {len(dw)}")
print(f"Semanas com viés negativo de {MAIN}: {(dw[f'vies_{MAIN}'] < 0).sum()} de {len(dw)}")
fig = graficos.erro_por_semana(dw, FOCUS); A.figura(fig, "diagnostico_erro_por_semana.png"); plt.show()

### 23.6–23.7 Maiores erros, concentração e picos

In [ ]:
cols = ["seller_id", "perfil", "week_start", "target_week", "revenue_current", "rev_mean_4", "zero_weeks_prev_4",
        "y"] + [f"pred_{m}" for m in FOCUS] + [f"err_{MAIN}", "pico"]
top = diag.nlargest(20, f"abs_{MAIN}")[list(dict.fromkeys(cols))].rename(columns={"y": "real"})
A.tabela(top, "diagnostico_maiores_erros.csv")
print(f"Dos 20 maiores erros de {MAIN}: {(top[f'err_{MAIN}'] < 0).sum()} subestimações, {int(top['pico'].sum())} em picos")
A.tabela(D.concentracao(diag, MODELS), "diagnostico_concentracao_erros.csv")

picos = D.peak_table(diag, diag["pico"], "global", MODELS, thr["PEAK_THRESHOLD"], thr["seller_peak_thr"])
picos_rel = D.peak_table(diag, diag["pico_relativo"], "relativo ao vendedor", MODELS, thr["PEAK_THRESHOLD"], thr["seller_peak_thr"])
A.tabela(picos, "diagnostico_picos.csv"); A.tabela(picos_rel, "diagnostico_picos_relativos.csv")
display(picos.drop(columns="criterio").set_index("modelo"))
order_pk = [m for m in ["LightGBM_MAE", "LightGBM_MAE_Exog", "CatBoost_MAE", "CatBoost_MAE_Exog", "CatBoost_MAE_raw",
                        "HistGB_MAE", "MediaMovel4", "Naive", "HistGB_Poisson", "CatBoost_Tweedie"] if m in MODELS]
fig = graficos.picos(picos, order_pk, thr["PEAK_THRESHOLD"], int(diag["pico"].sum()))
A.figura(fig, "diagnostico_picos.png"); plt.show()

In [ ]:
A.finalizar()